# VLMEvalKit Outputs 결과 확인

In [5]:
import re
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("outputs")

## 1. 전체 결과 로드

In [ ]:
score_files = sorted(OUTPUT_DIR.rglob("*_score.xlsx"))
all_dfs = {}
for f in score_files:
    key = str(f.relative_to(OUTPUT_DIR))
    df = pd.read_excel(f)
    all_dfs[key] = df
    print(f"{key}: {len(df)} samples, score={df['score'].mean()*100:.2f}%")

## 2. 정확도 요약

In [7]:
summary_rows = []
for key, df in all_dfs.items():
    correct = df["score"].sum()
    total = len(df)
    acc = correct / total * 100
    summary_rows.append({"file": key, "correct": int(correct), "total": total, "accuracy": f"{acc:.2f}%"})

summary_df = pd.DataFrame(summary_rows)
summary_df

,file,correct,total,accuracy
0,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1502,2174,69.09%
1,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1506,2174,69.27%
2,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1679,2700,62.19%
3,Qwen3-VL/Qwen3-VL-2B-Instruct/Qwen3-VL-2B-Inst...,1711,2700,63.37%
4,Qwen3-VL/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Inst...,1605,2174,73.83%
5,Qwen3-VL/Qwen3-VL-4B-Instruct/Qwen3-VL-4B-Inst...,1857,2700,68.78%


In [8]:
## 3. 결과 요약 테이블 (mf를 column으로)

rows = []
for f in score_files:
    rel = f.relative_to(OUTPUT_DIR)
    parts = rel.parts

    simple = "simple" if parts[0].endswith("-simple") else ""
    model = parts[1]

    m = re.search(r'_(Video-MME|MLVU)_tp(\d+)_mf(\d+)', f.stem)
    if not m:
        continue
    dataset = m.group(1)
    mf = int(m.group(3))

    key = str(rel)
    df = all_dfs[key]
    acc = df["score"].mean() * 100

    rows.append({
        "model": model,
        "dataset": dataset,
        "simple": simple,
        "mf": mf,
        "accuracy": round(acc, 2),
        "total": len(df),
    })

result_df = pd.DataFrame(rows)

# mf를 column으로 pivot
pivot = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="accuracy",
)
pivot.columns = [f"mf={c}" for c in pivot.columns]
pivot = pivot.reset_index()

# total 정보도 추가
pivot_total = result_df.pivot_table(
    index=["model", "dataset", "simple"],
    columns="mf",
    values="total",
)
pivot_total.columns = [f"n={c}" for c in pivot_total.columns]
pivot_total = pivot_total.reset_index()

final = pivot.merge(pivot_total, on=["model", "dataset", "simple"])
acc_cols = sorted([c for c in final.columns if c.startswith("mf=")], key=lambda x: int(x.split("=")[1]))
n_cols = sorted([c for c in final.columns if c.startswith("n=")], key=lambda x: int(x.split("=")[1]))
final = final[["model", "dataset", "simple"] + acc_cols + n_cols]
final

,model,dataset,simple,mf=256,mf=512,n=256,n=512
0,Qwen3-VL-2B-Instruct,MLVU,,69.09,69.27,2174.0,2174.0
1,Qwen3-VL-2B-Instruct,Video-MME,,62.19,63.37,2700.0,2700.0
2,Qwen3-VL-4B-Instruct,MLVU,,73.83,NaN,2174.0,NaN
3,Qwen3-VL-4B-Instruct,Video-MME,,68.78,NaN,2700.0,NaN
